# 📊 Visualization of Results (Sampled)

This notebook samples 500 rows from the Gold Layer results and visualizes them using Pandas and Matplotlib/Seaborn.
It connects to the same Nessie/Iceberg catalog as the Gold Layer pipeline.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style("whitegrid")

# Production Credentials (Same as GOLD_LAYER.ipynb)
PACKAGES = (
    "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,"
    "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,"
    "software.amazon.awssdk:bundle:2.17.178,"
    "software.amazon.awssdk:url-connection-client:2.17.178,"
    "org.apache.hadoop:hadoop-aws:3.3.1"
)

conf = (pyspark.SparkConf()
    .setAppName('Visualization-Notebook')
    .set('spark.jars.packages', PACKAGES)
    .set('spark.sql.extensions', 
         'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,'
         'org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
    .set('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog')
    .set('spark.sql.catalog.nessie.uri', 'http://140.238.224.207:19120/api/v1')
    .set('spark.sql.catalog.nessie.ref', 'main')
    .set('spark.sql.catalog.nessie.authentication.type', 'NONE')
    .set('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
    .set('spark.sql.catalog.nessie.warehouse', 's3a://lakehouse-prod/warehouse')
    .set('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .set('spark.sql.catalog.nessie.s3.endpoint', 'https://bmcfe6z38foz.compat.objectstorage.ap-mumbai-1.oraclecloud.com')
    .set('spark.hadoop.fs.s3a.access.key', '962c9f862226831e4edea90cfcfafb8a8dffcd51')
    .set('spark.hadoop.fs.s3a.secret.key', 'sd2rGU918DTmn35E4xJ8EV7BX2XUt7DkqC8v6WDNDUw=')
    .set('spark.hadoop.fs.s3a.endpoint', 'https://bmcfe6z38foz.compat.objectstorage.ap-mumbai-1.oraclecloud.com')
    .set('spark.hadoop.fs.s3a.path.style.access', 'true')
    .set('spark.hadoop.fs.s3a.connection.ssl.enabled', 'true')
    .set('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem'))

spark = SparkSession.builder.config(conf=conf).getOrCreate()
print("✅ Spark Connected for Visualization")

## 1. Customer Segments Visualization
Sampling 500 rows from `nessie.ecommerce.customer_segments_ml`.

In [ ]:
# Load data
df_segments = spark.table("nessie.ecommerce.customer_segments_ml")

# Sample 500 rows and convert to Pandas
pdf_segments = df_segments.limit(500).toPandas()

print(f"Loaded {len(pdf_segments)} rows for visualization.")
pdf_segments.head()

In [ ]:
# Plot Cluster Distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='cluster', data=pdf_segments, palette='viridis')
plt.title('Distribution of Customer Segments (Sample 500)')
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.show()

## 2. Churn Predictions Visualization
Sampling 500 rows from `nessie.ecommerce.churn_predictions_ml`.

In [ ]:
# Load data
df_churn = spark.table("nessie.ecommerce.churn_predictions_ml")

# Sample 500 rows and convert to Pandas
pdf_churn = df_churn.limit(500).toPandas()

print(f"Loaded {len(pdf_churn)} rows for visualization.")
pdf_churn.head()

In [ ]:
# Plot Churn Probability Distribution
plt.figure(figsize=(10, 6))
sns.histplot(pdf_churn['churn_probability'], bins=20, kde=True, color='red')
plt.title('Churn Probability Distribution (Sample 500)')
plt.xlabel('Churn Probability')
plt.ylabel('Count')
plt.axvline(0.7, color='k', linestyle='--', label='High Risk Threshold (0.7)')
plt.legend()
plt.show()

## 3. Product Recommendations Confidence
Sampling 500 rows from `nessie.ecommerce.product_recommendations_ml`.

In [ ]:
# Load data
df_recs = spark.table("nessie.ecommerce.product_recommendations_ml")

# Sample 500 rows
pdf_recs = df_recs.limit(500).toPandas()

print(f"Loaded {len(pdf_recs)} rows for visualization.")
pdf_recs.head()

In [ ]:
# Plot Confidence Distribution
plt.figure(figsize=(10, 6))
sns.histplot(pdf_recs['confidence'], bins=20, kde=True, color='green')
plt.title('Recommendation Confidence Score Distribution (Sample 500)')
plt.xlabel('Confidence')
plt.ylabel('Count')
plt.show()